In [1]:
import h5py
import numpy as np
from scipy.io import loadmat

In [2]:
def get_data(filename_real, filename_fake):
    
    normalizer = loadmat('/scratch/qy707/Mengze/mean_std_scale.mat')
    
    x_mean = normalizer['glb_mean'][0, 0, 1]
    x_std = normalizer['glb_std'][0, 0, 1]
    
    y_mean = normalizer['glb_mean'][0, 0, 0]
    y_std = normalizer['glb_std'][0, 0, 0]
    
    f_real = h5py.File(filename_real, "r")
    f_fake = h5py.File(filename_fake, "r")
    
    data_real = f_real['snapshots'][:]
    data_fake = f_fake['samples with various conditionals'][:]
    # real: (time * n_ens, 2, n_lat, n_lon)
    # fake: (time * n_ens, 1, n_lat, n_lon)
    
    n_lat = 96
    n_lon = 192
    
    x = data_real[:, 1, :, :].reshape(-1, 30, n_lat, n_lon) * (2 * x_std) + x_mean 
    y = data_real[:, 0, :, :].reshape(-1, 30, n_lat, n_lon) * (2 * y_std) + y_mean 
    y_hat = data_fake[:, 0, :, :].reshape(-1, 30, n_lat, n_lon) * (2 * y_std) + y_mean
    # (time, n_ens, n_lat, n_lon)
    
    f_real.close()
    f_fake.close()
    
    return x, y, y_hat

# Process Testing Data 1

In [3]:
filename_real = '/scratch/qy707/Mengze/data_0.0_2.0_1.1_0.0_context.hdf5'
filename_fake = '/scratch/qy707/Mengze/data_0.0_2.0_1.1_0.0_context_analysis.hdf5'
x, y, y_hat = get_data(filename_real, filename_fake)

N = x.shape[0]

x1 = x[np.arange(N) != 1957]
y1 = y[np.arange(N) != 1957]
y_hat1 = y_hat[np.arange(N) != 1957]

np.save('/scratch/qy707/Mengze/data1_y.npy', y1)
np.save('/scratch/qy707/Mengze/data1_y_hat.npy', y_hat1)

# Process Testing Data 2

In [4]:
filename_real = '/scratch/qy707/Mengze/data_0.0_2.0_1.2_0.0_context.hdf5'
filename_fake = '/scratch/qy707/Mengze/data_0.0_2.0_1.2_0.0_context_analysis.hdf5'
x2, y2, y_hat2 = get_data(filename_real, filename_fake)

np.save('/scratch/qy707/Mengze/data2_y.npy', y2)
np.save('/scratch/qy707/Mengze/data2_y_hat.npy', y_hat2)

# Loading Data

In [5]:
y1 = np.load('/scratch/qy707/Mengze/data1_y.npy')
y_hat1 = np.load('/scratch/qy707/Mengze/data1_y_hat.npy')

y2 = np.load('/scratch/qy707/Mengze/data2_y.npy')
y_hat2 = np.load('/scratch/qy707/Mengze/data2_y_hat.npy')

# Dim Reduction to 2 (two points)

In [6]:
filename_real = '/scratch/qy707/Mengze/data_0.0_2.0_1.1_0.0_context.hdf5'
f_real = h5py.File(filename_real, "r")

In [7]:
print(f_real.keys())

<KeysViewHDF5 ['lat', 'lon', 'snapshots']>


In [8]:
lat = f_real['lat'][:]

In [9]:
lon = f_real['lon'][:]

In [10]:
y1.shape

(3249, 30, 96, 192)

In [11]:
lat_mesh, lon_mesh = np.meshgrid(lat, lon, indexing='ij')

In [12]:
lat_mesh

array([[-88.57217 , -88.57217 , -88.57217 , ..., -88.57217 , -88.57217 ,
        -88.57217 ],
       [-86.722534, -86.722534, -86.722534, ..., -86.722534, -86.722534,
        -86.722534],
       [-84.86197 , -84.86197 , -84.86197 , ..., -84.86197 , -84.86197 ,
        -84.86197 ],
       ...,
       [ 84.86197 ,  84.86197 ,  84.86197 , ...,  84.86197 ,  84.86197 ,
         84.86197 ],
       [ 86.722534,  86.722534,  86.722534, ...,  86.722534,  86.722534,
         86.722534],
       [ 88.57217 ,  88.57217 ,  88.57217 , ...,  88.57217 ,  88.57217 ,
         88.57217 ]], shape=(96, 192), dtype=float32)

In [13]:
lon_mesh

array([[  0.   ,   1.875,   3.75 , ..., 354.375, 356.25 , 358.125],
       [  0.   ,   1.875,   3.75 , ..., 354.375, 356.25 , 358.125],
       [  0.   ,   1.875,   3.75 , ..., 354.375, 356.25 , 358.125],
       ...,
       [  0.   ,   1.875,   3.75 , ..., 354.375, 356.25 , 358.125],
       [  0.   ,   1.875,   3.75 , ..., 354.375, 356.25 , 358.125],
       [  0.   ,   1.875,   3.75 , ..., 354.375, 356.25 , 358.125]],
      shape=(96, 192), dtype=float32)

In [14]:
nyc_lat = 40.7128
nyc_lon = 360 - 74.0060

distance_sq = (lat_mesh - nyc_lat)**2 + (lon_mesh - nyc_lon)**2

nyc_index = np.unravel_index(np.argmin(distance_sq), lat_mesh.shape)

print("NYC Index:", nyc_index)
print("Latitude at index:", lat_mesh[nyc_index])
print("Longitude at index:", lon_mesh[nyc_index])

NYC Index: (np.int64(69), np.int64(153))
Latitude at index: 40.102978
Longitude at index: 286.875


In [15]:
bos_lat = 42.3555
bos_lon = 360 - 71.0565

distance_sq = (lat_mesh - bos_lat)**2 + (lon_mesh - bos_lon)**2

bos_index = np.unravel_index(np.argmin(distance_sq), lat_mesh.shape)

print("BOS Index:", bos_index)
print("Latitude at index:", lat_mesh[bos_index])
print("Longitude at index:", lon_mesh[bos_index])

BOS Index: (np.int64(70), np.int64(154))
Latitude at index: 41.96822
Longitude at index: 288.75


In [16]:
mm_lat = 25.7617
mm_lon = 360 - 80.1918

distance_sq = (lat_mesh - mm_lat)**2 + (lon_mesh - mm_lon)**2

mm_index = np.unravel_index(np.argmin(distance_sq), lat_mesh.shape)

print("Miami Index:", mm_index)
print("Latitude at index:", lat_mesh[mm_index])
print("Longitude at index:", lon_mesh[mm_index])

Miami Index: (np.int64(61), np.int64(149))
Latitude at index: 25.180986
Longitude at index: 279.375


In [17]:
ol_lat = 28.5384
ol_lon = 360 - 81.3789

distance_sq = (lat_mesh - ol_lat)**2 + (lon_mesh - ol_lon)**2

ol_index = np.unravel_index(np.argmin(distance_sq), lat_mesh.shape)

print("Orlando Index:", ol_index)
print("Latitude at index:", lat_mesh[ol_index])
print("Longitude at index:", lon_mesh[ol_index])

Orlando Index: (np.int64(63), np.int64(149))
Latitude at index: 28.911493
Longitude at index: 279.375


In [18]:
kw_lat = 24.5551
kw_lon = 360 - 81.7800

distance_sq = (lat_mesh - kw_lat)**2 + (lon_mesh - kw_lon)**2

kw_index = np.unravel_index(np.argmin(distance_sq), lat_mesh.shape)

print("Key West Index:", kw_index)
print("Latitude at index:", lat_mesh[kw_index])
print("Longitude at index:", lon_mesh[kw_index])

Key West Index: (np.int64(61), np.int64(148))
Latitude at index: 25.180986
Longitude at index: 277.5


In [19]:
def Pick_Two_Points(y, idx1, idx2):
    
    # y: (n_batch, n_samples, n_lat, n_lon)
    
    y_p1 = y[:, :, idx1[0], idx1[1]]
    y_p2 = y[:, :, idx2[0], idx2[1]]
    # (n_batch, n_samples)
    
    y_reduce = np.stack((y_p1, y_p2), axis=-1)  
    # Shape: (n_batch, n_samples, 2)
    
    return y_reduce

In [20]:
idx1 = kw_index
idx2 = mm_index

y1_reduce = Pick_Two_Points(y1, idx1, idx2)
y_hat1_reduce = Pick_Two_Points(y_hat1, idx1, idx2)

y2_reduce = Pick_Two_Points(y2, idx1, idx2)
y_hat2_reduce = Pick_Two_Points(y_hat2, idx1, idx2)

In [21]:
np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y1_reduce_TP.npy', y1_reduce)
np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y_hat1_reduce_TP.npy', y_hat1_reduce)

np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y2_reduce_TP.npy', y2_reduce)
np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y_hat2_reduce_TP.npy', y_hat2_reduce)

# Dim Reduction to 2 (mean)

In [22]:
def North_South_Mean(y):
    
    # y: (n_batch, n_samples, n_lat, n_lon)
    
    n_lat = 96
    
    y_north = y[:, :, (n_lat // 2):, :]
    y_south = y[:, :, :(n_lat // 2), :]
    
    y_north_mean = np.mean(y_north, axis=(2, 3))
    y_south_mean = np.mean(y_south, axis=(2, 3))
    # (n_batch, n_samples)
    
    y_reduce = np.stack((y_north_mean, y_south_mean), axis=-1)  
    # Shape: (n_batch, n_samples, 2)
    
    return y_reduce

In [23]:
y1_reduce = North_South_Mean(y1)
y_hat1_reduce = North_South_Mean(y_hat1)

y2_reduce = North_South_Mean(y2)
y_hat2_reduce = North_South_Mean(y_hat2)

In [24]:
np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y1_reduce_NSM.npy', y1_reduce)
np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y_hat1_reduce_NSM.npy', y_hat1_reduce)

np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y2_reduce_NSM.npy', y2_reduce)
np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y_hat2_reduce_NSM.npy', y_hat2_reduce)

# Dim Reduction to 3

In [25]:
def North_Middle_South_Mean(y):
    
    # y: (n_batch, n_samples, n_lat, n_lon)
    
    n_lat = 96
    
    y_north = y[:, :, (n_lat // 3) * 2:, :]
    y_middle = y[:, :, (n_lat // 3):(n_lat // 3) * 2, :]
    y_south = y[:, :, :(n_lat // 3), :]
    
    y_north_mean = np.mean(y_north, axis=(2, 3))
    y_middle_mean = np.mean(y_middle, axis=(2, 3))
    y_south_mean = np.mean(y_south, axis=(2, 3))
    # (n_batch, n_samples)
    
    y_reduce = np.stack((y_north_mean, y_middle_mean, y_south_mean), axis=-1)  
    # Shape: (n_batch, n_samples, 3)
    
    return y_reduce

In [26]:
y1_reduce = North_Middle_South_Mean(y1)
y_hat1_reduce = North_Middle_South_Mean(y_hat1)

y2_reduce = North_Middle_South_Mean(y2)
y_hat2_reduce = North_Middle_South_Mean(y_hat2)

np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y1_reduce_NMSM.npy', y1_reduce)
np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y_hat1_reduce_NMSM.npy', y_hat1_reduce)

np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y2_reduce_NMSM.npy', y2_reduce)
np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y_hat2_reduce_NMSM.npy', y_hat2_reduce)

# Dim Reduction to 4

In [27]:
def Quarter_Mean(y):
    
    # y: (n_batch, n_samples, n_lat, n_lon)
    
    n_lat = 96
    n_lon = 192
    
    y_north_west = y[:, :, (n_lat // 2):, :(n_lon // 2)]
    y_north_east = y[:, :, (n_lat // 2):, (n_lon // 2):]
    
    y_south_west = y[:, :, :(n_lat // 2), :(n_lon // 2)]
    y_south_east = y[:, :, :(n_lat // 2), (n_lon // 2):]
    
    y_north_west_mean = np.mean(y_north_west, axis=(2, 3))
    y_south_west_mean = np.mean(y_south_west, axis=(2, 3))
    y_north_east_mean = np.mean(y_north_east, axis=(2, 3))
    y_south_east_mean = np.mean(y_south_east, axis=(2, 3))
    # (n_batch, n_samples)
    
    y_reduce = np.stack((y_north_west_mean, y_south_west_mean, y_north_east_mean, y_south_east_mean), axis=-1)  
    # Shape: (n_batch, n_samples, 4)
    
    return y_reduce

In [28]:
y1_reduce = Quarter_Mean(y1)
y_hat1_reduce = Quarter_Mean(y_hat1)

y2_reduce = Quarter_Mean(y2)
y_hat2_reduce = Quarter_Mean(y_hat2)

np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y1_reduce_QM.npy', y1_reduce)
np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y_hat1_reduce_QM.npy', y_hat1_reduce)

np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y2_reduce_QM.npy', y2_reduce)
np.save('/home/qy707/CP4GenerativeModel/data/Mengze/y_hat2_reduce_QM.npy', y_hat2_reduce)